In [99]:
import pandas as pd
import numpy as np
import os
from glob import glob
from tqdm.auto import tqdm
import pickle
import h5py

def load_data(filename):
    with h5py.File(filename, 'r') as f:
        X_train = f['X_train'][:]
        y_train = f['y_train'][:]
        X_test = f['X_test'][:]
        y_test = f['y_test'][:]

    result = {
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
        'y_test': y_test,
    }

    return result['X_train'], result['X_test'], result['y_train'], result['y_test']

In [2]:
df = pd.read_csv("df_best.csv")

df

,dataset_name,model_name,learning_rate,patience,max_depth,max_n_iter,feature_order,recompute_feature_order,reuse_features,use_fast,...,l1_ratio,dual,num_rules,rfmode,max_rules,memory_par,model_type,max_trees,min_impurity_decrease,max_features
0,adult,EBM,0.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,adult,EBMLasso,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,adult,EBMSweep,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,adult,FIGS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,48.0,NaN,NaN,NaN,0.0,NaN
4,adult,LogisticGAM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,titanic,lgbm,0.10,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
237,titanic,log_reg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.5,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
238,titanic,ridge,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
239,titanic,rulefit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,classification,100.0,0.1,r,NaN,NaN,NaN


In [4]:
df_sub = df.groupby('model_name').first().reset_index()

df_sub

,model_name,dataset_name,learning_rate,patience,max_depth,max_n_iter,feature_order,recompute_feature_order,reuse_features,use_fast,...,l1_ratio,dual,num_rules,rfmode,max_rules,memory_par,model_type,max_trees,min_impurity_decrease,max_features
0,EBM,adult,0.05,NaN,NaN,NaN,None,None,None,None,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN
1,EBMLasso,adult,0.01,NaN,NaN,NaN,None,None,None,None,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN
2,EBMSweep,adult,0.01,NaN,NaN,NaN,None,None,None,None,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN
3,FIGS,adult,NaN,NaN,NaN,NaN,None,None,None,None,...,NaN,None,NaN,None,48.0,NaN,None,NaN,0.0,NaN
4,LogisticGAM,adult,NaN,NaN,NaN,NaN,None,None,None,None,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN
5,RF,adult,NaN,NaN,10.0,NaN,None,None,None,None,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN
6,RuleCard,adult,0.50,15.0,3.0,500.0,random,False,True,True,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN
7,ScoreCard,adult,0.50,15.0,3.0,500.0,random,False,True,True,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN
8,ScoreCard2,adult,0.50,15.0,3.0,500.0,random,False,True,True,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN
9,ScoreCard3,adult,0.50,15.0,3.0,500.0,random,False,True,True,...,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN


In [62]:
df_sub.model_name.values

array(['EBM', 'EBMLasso', 'EBMSweep', 'FIGS', 'LogisticGAM', 'RF',
       'RuleCard', 'ScoreCard', 'ScoreCard2', 'ScoreCard3', 'TreeGAMCl',
       'apriori_log', 'apriori_ridge', 'boley', 'dt', 'fasterrisk',
       'lgbm', 'log_reg', 'ridge', 'rulefit', 'xgboost'], dtype=object)

In [30]:
thr_feat_imp = 10**-4

# EBM, EBMSweep, EBMLasso

In [45]:
#Local: #feat+#inter
#Global: \sum(#bins per feature)

def explain_ebm(ebm, X=None):
    gl = 0
    lo = np.sum(np.array(ebm.explain_global().data()['scores']) > thr_feat_imp)
    n_bins_feat = []
    bins_features = {f'feature_{i:04d}': [len(x_)+1 for x_ in x] for i, x in enumerate(ebm.bins_)}
    for feat_name, feat_score in zip(ebm.explain_global().data()['names'], ebm.explain_global().data()['scores']):
        if feat_score <= thr_feat_imp:
            continue
        
        if '&' not in feat_name:
            gl += bins_features[feat_name][0]
        else:
            names = feat_name.split(' & ')
            gl += bins_features[names[0]][-1] + bins_features[names[1]][-1]
    return gl, lo
    
ebm = pickle.load(open(df_sub[df_sub.model_name == 'EBM'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))

explain_ebm(ebm)

(3755, 136)

# LGBM

In [55]:
def lgbm_leaf_depths(tree, depth=0):
    if "leaf_index" in tree:
        return [depth]
    
    left = lgbm_leaf_depths(tree["left_child"], depth + 1)
    right = lgbm_leaf_depths(tree["right_child"], depth + 1)
    return left + right

def lgbm_nodes(tree:dict):
    if 'leaf_index' in tree:
        return 0
    else:
        return 1 + lgbm_nodes(tree['left_child']) + lgbm_nodes(tree['right_child'])

def lgbm_rule_nodes(tree, depth=0):
    if "leaf_index" in tree:
        return [depth]
    
    return lgbm_rule_nodes(tree["left_child"], depth + 1) + lgbm_rule_nodes(tree["right_child"], depth + 1)

def explain_lgbm(lgbm, X=None):
    gl = np.sum([np.sum(lgbm_rule_nodes(t['tree_structure'])) for t in lgbm.booster_.dump_model()['tree_info']])
    lo = np.sum([np.mean(lgbm_leaf_depths(t['tree_structure'])) for t in lgbm.booster_.dump_model()['tree_info']])

    return gl, lo


lgbm = pickle.load(open(df_sub[df_sub.model_name == 'lgbm'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))

explain_lgbm(lgbm)

(22534, 2944.9142857142856)

# LogisticGAM

In [71]:
def explain_lgam(lgam, X=None):
    gl = sum(np.abs(lgam.coef_)[:-1] > thr_feat_imp)
    lo = len(lgam.terms)-1

    return gl, lo

lgam = pickle.load(open(df_sub[df_sub.model_name == 'LogisticGAM'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))

explain_lgam(lgam)

(680, 34)

# RuleFit

In [101]:
def explain_rfit(rfit, X=None):
    gl = np.sum(np.abs(rfit.coef_) > thr_feat_imp)
    lo = np.mean(np.sum(rfit.rule_ensemble.transform(X)[:, np.abs(rfit.coef_) > thr_feat_imp], axis=1))

    return gl, lo

X_train, X_test, y_train, y_test = load_data(df_sub[df_sub.model_name == 'rulefit'].filename.iloc[0].replace('.csv', '.h5'))
rfit = pickle.load(open(df_sub[df_sub.model_name == 'rulefit'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))

explain_rfit(rfit, X_train)

(351, 76.68206234231302)

In [76]:
len(rfit.coef_)

1374

In [84]:
np.sum(np.abs(rfit.coef_) > thr_feat_imp)

351

In [77]:
len(rfit.get_rules())

1374

In [78]:
rfit.get_rules()

,rule,type,coef,support,importance
0,feature_21 <= 5095.5,rule,0.000000,0.945192,0.000000
1,feature_21 > 5095.5,rule,0.000000,0.054808,0.000000
2,feature_28 <= 1.5 & feature_22 <= 12.5 & featu...,rule,-0.050660,0.194231,0.020042
3,feature_28 <= 1.5 & feature_22 <= 12.5 & featu...,rule,0.000000,0.107692,0.000000
4,feature_28 <= 1.5 & feature_22 <= 12.5 & featu...,rule,0.000000,0.007692,0.000000
...,...,...,...,...,...
1369,feature_12 <= 51.5,rule,0.000000,0.834615,0.000000
1370,feature_12 > 51.5 & feature_22 <= 9.5 & featur...,rule,0.094027,0.082692,0.025897
1371,feature_12 > 51.5 & feature_22 <= 9.5 & featur...,rule,0.158308,0.000962,0.004907
1372,feature_12 > 51.5 & feature_22 > 9.5 & feature...,rule,0.000000,0.015385,0.000000


In [81]:
len(rfit.tree_generator.estimators_)

552

In [87]:
rfit.get_rules()[rfit.get_rules().importance > thr_feat_imp]

,rule,type,coef,support,importance
2,feature_28 <= 1.5 & feature_22 <= 12.5 & featu...,rule,-0.050660,0.194231,0.020042
9,feature_28 > 1.5 & feature_21 > 7731.5,rule,0.292489,0.005769,0.022152
11,feature_28 <= 1.5 & feature_21 > 5095.5,rule,3.407996,0.040385,0.670897
18,feature_28 <= 1.5 & feature_22 <= 11.5 & featu...,rule,-0.161685,0.292308,0.073538
23,feature_28 > 1.5 & feature_21 > 7790.0,rule,0.189604,0.010577,0.019396
...,...,...,...,...,...
1357,feature_17 <= 0.5 & feature_21 <= 7073.5 & fea...,rule,-0.042261,0.185577,0.016429
1363,feature_1 > 0.5 & feature_25 <= 0.5 & feature_...,rule,0.340072,0.460577,0.169506
1367,feature_12 > 47.5 & feature_22 <= 10.5,rule,0.121647,0.157692,0.044335
1370,feature_12 > 51.5 & feature_22 <= 9.5 & featur...,rule,0.094027,0.082692,0.025897


In [98]:
np.mean(np.sum(rfit.rule_ensemble.transform(X_train)[:, np.abs(rfit.coef_) > thr_feat_imp], axis=1))

76.68206234231302

# LogisticRegressor

In [121]:
def explain_lReg(lReg, X=None):
    gl = np.sum(np.abs(lReg.coef_) > thr_feat_imp)
    lo = np.mean(np.sum((X * lReg.coef_) > thr_feat_imp, axis=1))

    return gl, lo

lReg = pickle.load(open(df_sub[df_sub.model_name == 'log_reg'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))
print(*explain_lReg(lReg, X_train))

lReg = pickle.load(open(df_sub[df_sub.model_name == 'ridge'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))
print(*explain_lReg(lReg, X_train))

34 4.358020672255229
33 4.644787173435338


In [102]:
lReg = pickle.load(open(df_sub[df_sub.model_name == 'log_reg'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))

lReg

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.5
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [115]:
np.mean(np.sum((X_test * lReg.coef_) > thr_feat_imp, axis=1))

4.359049479166667

In [118]:
np.sum(np.abs(lReg.coef_) > thr_feat_imp)

34

# TreeGAM

In [132]:
#Local: #feat+#iter
#Global: \sum(#bins per feature)

def explain_tgam(tgam, X=None):
    gl = np.sum([tree.tree_.n_leaves for tree in tgam.estimators_])
    lo = tgam.estimators_[0].max_features_

    return gl, lo

tgam = pickle.load(open(df_sub[df_sub.model_name == 'TreeGAMCl'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))

explain_tgam(tgam)

(12900, 34)

In [129]:
tgam.estimators_[0].max_features_

34

In [131]:
X_train.shape

(24574, 34)

# Boley

In [194]:
def explain_boley(boley, X=None):
    df = pd.DataFrame(X_train, columns=[f'feat{i}' for i in range(X_train.shape[1])])
    gl = len(boley.rules_)
    lo = np.mean(np.sum(np.vstack([boley.rules_.members[i](df) != 0 for i in range(len(boley.rules_))]).T, axis=1))

    return gl, lo

boley = pickle.load(open(df_sub[df_sub.model_name == 'boley'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))

explain_boley(boley)

(200, 1.0793521608203793)

In [135]:
len(boley.rules_)

200

In [142]:
boley.rules_[0]

   -0.8954 if feat19>=2.0 & feat7<=13.0

In [186]:
df = pd.DataFrame(X_train, columns=[f'feat{i}' for i in range(X_train.shape[1])])

boley.rules_(df).shape

(24574,)

In [187]:
X_train.shape

(24574, 34)

In [191]:
np.vstack([boley.rules_.members[i](df) for i in range(len(boley.rules_))]).T.shape

(24574, 200)

In [193]:
np.sum(np.vstack([boley.rules_.members[i](df) != 0 for i in range(len(boley.rules_))]).T, axis=1).shape

(24574,)

In [190]:
np.mean(np.sum(np.vstack([boley.rules_.members[i](df) != 0 for i in range(len(boley.rules_))]).T, axis=1))

1.0793521608203793

In [173]:
boley.rules_.members[0](df).shape

(24574,)

In [179]:
np.sum(np.abs(boley.rules_.members[10](df)))

0.0

# Apriori

In [206]:
def explain_apriori(apriori, X=None):
    X_rules = apriori._encode(pd.DataFrame(X, columns=[f'feat_{i}' for i in range(X.shape[1])]).infer_objects())
    gl = np.sum(np.abs(apriori.clf.coef_) > thr_feat_imp)
    lo = np.mean(np.sum((X_rules * apriori.clf.coef_) > thr_feat_imp, axis=1))

    return gl, lo

apriori = pickle.load(open(df_sub[df_sub.model_name == 'apriori_log'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))
explain_apriori(apriori, X_train)

(39, 0.0)

In [215]:
apriori = pickle.load(open(df_sub[df_sub.model_name == 'apriori_ridge'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))
explain_apriori(apriori, X_train)

(3198, 0.0)

# Random Forest

In [218]:
from sklearn.tree import _tree

def rf_leaf_depths(tree):
    cl, cr = tree.children_left, tree.children_right

    def visit(node=0, depth=0):
        if cl[node] == _tree.TREE_LEAF:
            return [depth]
        return visit(cl[node], depth + 1) + visit(cr[node], depth + 1)

    return visit()

def explain_rf(rf, X=None):
    depths = [rf_leaf_depths(est.tree_) for est in rf.estimators_]

    gl = np.sum([np.sum(d) for d in depths])
    lo = np.sum([np.mean(d) for d in depths])

    return gl, lo

rf = pickle.load(open(df_sub[df_sub.model_name == 'RF'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))
explain_rf(rf, X_train)

(9282972, 2212.964679106203)

# XGBoost

In [222]:
def xgb_leaf_depths(tree, depth=0):
    if "leaf" in tree:
        return [depth]

    return (
        xgb_leaf_depths(tree["children"][0], depth + 1) +
        xgb_leaf_depths(tree["children"][1], depth + 1)
    )

def xgb_rule_nodes(tree, depth=0):
    if "leaf" in tree:
        return [depth]

    return (
        xgb_rule_nodes(tree["children"][0], depth + 1) +
        xgb_rule_nodes(tree["children"][1], depth + 1)
    )

def explain_xgb(xgb_model, X=None):
    dump = xgb_model.get_booster().get_dump(with_stats=False, dump_format="json")
    trees = [eval(t) for t in dump]  # XGBoost JSON -> dict

    gl = np.sum([np.sum(xgb_rule_nodes(t)) for t in trees])
    lo = np.sum([np.mean(xgb_leaf_depths(t)) for t in trees])

    return gl, lo

xgboost = pickle.load(open(df_sub[df_sub.model_name == 'xgboost'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))
explain_xgb(xgboost, X_train)

(20784, 2870.8976190476187)

In [224]:
figs = pickle.load(open(df_sub[df_sub.model_name == 'FIGS'].filename.iloc[0].replace('.csv', '.pkl'), 'rb'))



,max_rules,48
,max_trees,None
,min_impurity_decrease,0.0
,random_state,42
,max_features,None
,max_depth,None


In [225]:
figs.trees_

[X_5 <= 1.500 (Tree #0 root),
 X_33 <= 5119.000 (Tree #1 root),
 X_21 <= 41.500 (Tree #2 root),
 X_29 <= 2384.500 (Tree #3 root),
 X_23 <= 0.500 (Tree #4 root),
 X_14 <= 0.500 (Tree #5 root),
 X_2 <= 0.500 (Tree #6 root)]

In [228]:
figs.trees_[0].left

X_29 <= 1782.500 (split)